# Stage 1 – 1‑cycle vs Unconstrained Comparisons


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# --- Config -----------------------------------------------------------------
YEAR = 1985

# Pair A: 1‑month power (unconstrained vs 1‑cycle)
PAIR_A = {
    "unconstrained": "stagel_3month_1month_cap",
    "cycle_1": "stagel_1month_1cycle",
}

# Pair B: 3‑month power (unconstrained vs 1‑cycle)
PAIR_B = {
    "unconstrained": "stagel_3month_cap",
    "cycle_1": "stagel_3month_1cycle",
}

print("Year:", YEAR)
print("Pair A:", PAIR_A)
print("Pair B:", PAIR_B)


In [ ]:

# --- Helpers ----------------------------------------------------------------

def _strip_tz(idx):
    if hasattr(idx, 'tz') and idx.tz is not None:
        return idx.tz_convert(None)
    return idx


def tidy_storage_df(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    meta_cols = [c for c in ["bus_id", "asset_type", "zone", "is_seasonal"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    tidy[value_name] = pd.to_numeric(tidy[value_name], errors="coerce").fillna(0.0)
    return tidy


def tidy_bus_df(path: Path, value_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    meta_cols = [c for c in ["bus_id", "zone"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    tidy[value_name] = pd.to_numeric(tidy[value_name], errors="coerce").fillna(0.0)
    return tidy


def total_ts(df: pd.DataFrame, value_name: str) -> pd.Series:
    return df.groupby("timestamp")[value_name].sum()


def load_seasonal_soc(run_dir: Path, year: int) -> pd.Series:
    p = run_dir / f"storage_state_seasonal_{year}.csv"
    if not p.exists():
        return pd.Series(dtype=float)
    df = pd.read_csv(p)
    if df.empty:
        return pd.Series(dtype=float)
    cols = [c for c in df.columns if c not in ("zone",)]
    df = df[cols]
    if not df.empty and df.iloc[0, 0] == "bus_id":
        df = df.iloc[1:]
    value_cols = [c for c in df.columns if c != "bus_id"]
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
    soc = df[value_cols].sum(axis=0)
    soc.index = pd.to_datetime(soc.index, errors="coerce")
    soc = soc.dropna()
    if hasattr(soc.index, 'tz') and soc.index.tz is not None:
        soc.index = soc.index.tz_convert(None)
    return soc


def safe_sum(series: pd.Series) -> float:
    return pd.to_numeric(series, errors="coerce").fillna(0.0).sum()


In [ ]:

# --- Output root (robust) ----------------------------------------------------
output_root = Path("acorn-julia/runs/low_RE_mod_elec_iter0/outputs/historical_1980_2019")
if not output_root.exists():
    output_root = Path.cwd().parent / "outputs" / "historical_1980_2019"
print("output_root:", output_root)


## A) Pair A: 1‑month power (unconstrained vs 1‑cycle)


In [ ]:

# Load Pair A
run_data = {}
for label, name in PAIR_A.items():
    run_dir = output_root / name
    run_data[label] = {
        "charge_base": tidy_storage_df(pd.read_csv(run_dir / f"charge_base_{YEAR}.csv"), "charge"),
        "discharge_base": tidy_storage_df(pd.read_csv(run_dir / f"discharge_base_{YEAR}.csv"), "discharge"),
        "charge_seasonal": tidy_storage_df(pd.read_csv(run_dir / f"charge_seasonal_{YEAR}.csv"), "charge"),
        "discharge_seasonal": tidy_storage_df(pd.read_csv(run_dir / f"discharge_seasonal_{YEAR}.csv"), "discharge"),
        "load_shed": tidy_bus_df(run_dir / f"load_shedding_{YEAR}.csv", "load_shedding"),
        "wind_curt": tidy_bus_df(run_dir / f"wind_curtailment_{YEAR}.csv", "wind_curtailment"),
        "solar_curt": tidy_bus_df(run_dir / f"solar_curtailment_{YEAR}.csv", "solar_curtailment"),
        "soc_seasonal": load_seasonal_soc(run_dir, YEAR),
    }

print("Loaded Pair A:", list(run_data.keys()))


In [ ]:

# Summary table
rows = []
for label, data in run_data.items():
    base_dis = total_ts(data["discharge_base"], "discharge")
    seas_dis = total_ts(data["discharge_seasonal"], "discharge")
    base_ch = total_ts(data["charge_base"], "charge")
    seas_ch = total_ts(data["charge_seasonal"], "charge")
    ls = total_ts(data["load_shed"], "load_shedding")
    wind = total_ts(data["wind_curt"], "wind_curtailment")
    solar = total_ts(data["solar_curt"], "solar_curtailment")

    rows.append({
        "run": label,
        "base_charge_MWh": safe_sum(base_ch),
        "base_discharge_MWh": safe_sum(base_dis),
        "seasonal_charge_MWh": safe_sum(seas_ch),
        "seasonal_discharge_MWh": safe_sum(seas_dis),
        "load_shed_MWh": safe_sum(ls),
        "wind_curt_MWh": safe_sum(wind),
        "solar_curt_MWh": safe_sum(solar),
    })

summary = pd.DataFrame(rows)
for c in summary.columns[1:]:
    summary[c] = summary[c].round(2)

display(summary)


In [ ]:

# Pairwise plots (discharge + load shed)
labels = list(PAIR_A.keys())
a, b = labels[0], labels[1]

for title in ["Pair A"]:
    a_dis = total_ts(run_data[a]["discharge_base"], "discharge") + total_ts(run_data[a]["discharge_seasonal"], "discharge")
    b_dis = total_ts(run_data[b]["discharge_base"], "discharge") + total_ts(run_data[b]["discharge_seasonal"], "discharge")

    a_ls = total_ts(run_data[a]["load_shed"], "load_shedding")
    b_ls = total_ts(run_data[b]["load_shed"], "load_shedding")

    fig, axes = plt.subplots(2, 1, figsize=(12,6), sharex=True)
    axes[0].plot(a_dis.index, a_dis.values, label=f"{a} discharge", alpha=0.7)
    axes[0].plot(b_dis.index, b_dis.values, label=f"{b} discharge", alpha=0.7)
    axes[0].set_title(f"{title}: total discharge")
    axes[0].legend()

    axes[1].plot(a_ls.index, a_ls.values, label=f"{a} load shed", alpha=0.7)
    axes[1].plot(b_ls.index, b_ls.values, label=f"{b} load shed", alpha=0.7)
    axes[1].set_title(f"{title}: load shedding")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:

# Curtailment comparison
labels = list(PAIR_A.keys())
a, b = labels[0], labels[1]

aw = total_ts(run_data[a]["wind_curt"], "wind_curtailment")
bw = total_ts(run_data[b]["wind_curt"], "wind_curtailment")
aso = total_ts(run_data[a]["solar_curt"], "solar_curtailment")
bs = total_ts(run_data[b]["solar_curt"], "solar_curtailment")

fig, axes = plt.subplots(2, 1, figsize=(12,6), sharex=True)
axes[0].plot(aw.index, aw.values, label=f"{a} wind", alpha=0.7)
axes[0].plot(bw.index, bw.values, label=f"{b} wind", alpha=0.7)
axes[0].set_title("Pair A: wind curtailment")
axes[0].legend()

axes[1].plot(aso.index, aso.values, label=f"{a} solar", alpha=0.7)
axes[1].plot(bs.index, bs.values, label=f"{b} solar", alpha=0.7)
axes[1].set_title("Pair A: solar curtailment")
axes[1].legend()

plt.tight_layout()
plt.show()


## B) Pair B: 3‑month power (unconstrained vs 1‑cycle)


In [ ]:

# Load Pair B
run_data = {}
for label, name in PAIR_B.items():
    run_dir = output_root / name
    run_data[label] = {
        "charge_base": tidy_storage_df(pd.read_csv(run_dir / f"charge_base_{YEAR}.csv"), "charge"),
        "discharge_base": tidy_storage_df(pd.read_csv(run_dir / f"discharge_base_{YEAR}.csv"), "discharge"),
        "charge_seasonal": tidy_storage_df(pd.read_csv(run_dir / f"charge_seasonal_{YEAR}.csv"), "charge"),
        "discharge_seasonal": tidy_storage_df(pd.read_csv(run_dir / f"discharge_seasonal_{YEAR}.csv"), "discharge"),
        "load_shed": tidy_bus_df(run_dir / f"load_shedding_{YEAR}.csv", "load_shedding"),
        "wind_curt": tidy_bus_df(run_dir / f"wind_curtailment_{YEAR}.csv", "wind_curtailment"),
        "solar_curt": tidy_bus_df(run_dir / f"solar_curtailment_{YEAR}.csv", "solar_curtailment"),
        "soc_seasonal": load_seasonal_soc(run_dir, YEAR),
    }

print("Loaded Pair B:", list(run_data.keys()))


In [ ]:

# Summary table
rows = []
for label, data in run_data.items():
    base_dis = total_ts(data["discharge_base"], "discharge")
    seas_dis = total_ts(data["discharge_seasonal"], "discharge")
    base_ch = total_ts(data["charge_base"], "charge")
    seas_ch = total_ts(data["charge_seasonal"], "charge")
    ls = total_ts(data["load_shed"], "load_shedding")
    wind = total_ts(data["wind_curt"], "wind_curtailment")
    solar = total_ts(data["solar_curt"], "solar_curtailment")

    rows.append({
        "run": label,
        "base_charge_MWh": safe_sum(base_ch),
        "base_discharge_MWh": safe_sum(base_dis),
        "seasonal_charge_MWh": safe_sum(seas_ch),
        "seasonal_discharge_MWh": safe_sum(seas_dis),
        "load_shed_MWh": safe_sum(ls),
        "wind_curt_MWh": safe_sum(wind),
        "solar_curt_MWh": safe_sum(solar),
    })

summary = pd.DataFrame(rows)
for c in summary.columns[1:]:
    summary[c] = summary[c].round(2)

display(summary)


In [ ]:

# Pairwise plots (discharge + load shed)
labels = list(PAIR_B.keys())
a, b = labels[0], labels[1]

for title in ["Pair B"]:
    a_dis = total_ts(run_data[a]["discharge_base"], "discharge") + total_ts(run_data[a]["discharge_seasonal"], "discharge")
    b_dis = total_ts(run_data[b]["discharge_base"], "discharge") + total_ts(run_data[b]["discharge_seasonal"], "discharge")

    a_ls = total_ts(run_data[a]["load_shed"], "load_shedding")
    b_ls = total_ts(run_data[b]["load_shed"], "load_shedding")

    fig, axes = plt.subplots(2, 1, figsize=(12,6), sharex=True)
    axes[0].plot(a_dis.index, a_dis.values, label=f"{a} discharge", alpha=0.7)
    axes[0].plot(b_dis.index, b_dis.values, label=f"{b} discharge", alpha=0.7)
    axes[0].set_title(f"{title}: total discharge")
    axes[0].legend()

    axes[1].plot(a_ls.index, a_ls.values, label=f"{a} load shed", alpha=0.7)
    axes[1].plot(b_ls.index, b_ls.values, label=f"{b} load shed", alpha=0.7)
    axes[1].set_title(f"{title}: load shedding")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:

# Curtailment comparison
labels = list(PAIR_B.keys())
a, b = labels[0], labels[1]

aw = total_ts(run_data[a]["wind_curt"], "wind_curtailment")
bw = total_ts(run_data[b]["wind_curt"], "wind_curtailment")
aso = total_ts(run_data[a]["solar_curt"], "solar_curtailment")
bs = total_ts(run_data[b]["solar_curt"], "solar_curtailment")

fig, axes = plt.subplots(2, 1, figsize=(12,6), sharex=True)
axes[0].plot(aw.index, aw.values, label=f"{a} wind", alpha=0.7)
axes[0].plot(bw.index, bw.values, label=f"{b} wind", alpha=0.7)
axes[0].set_title("Pair B: wind curtailment")
axes[0].legend()

axes[1].plot(aso.index, aso.values, label=f"{a} solar", alpha=0.7)
axes[1].plot(bs.index, bs.values, label=f"{b} solar", alpha=0.7)
axes[1].set_title("Pair B: solar curtailment")
axes[1].legend()

plt.tight_layout()
plt.show()


## C) Monthly net energy + SOC (optional)


In [ ]:

# If you want monthly seasonality for each pair, re-run this after loading a pair.
